# V4 — full frozen NeuroLM/GPT-2 verbalizer → ZuCo sentiment

This final broad-screen notebook tests the complete NeuroLM-B EEG-to-GPT-2 route. It uses the same fixed sentiment instruction for every recording, never supplies the stimulus sentence, and scores the fixed GPT-2 label tokens `negative`, `neutral`, and `positive` through one small residual adapter.

Run V2 through its raw-cache cell first. V4 reuses those packs and the existing 2.38 GB checkpoint. Select a **GPU** runtime and run every cell in order.

In [ ]:
# 1) Fetch this project, install small Colab-only dependencies, and pin the official source.
from pathlib import Path
import importlib.metadata as package_metadata
import os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")
UPSTREAM_URL = "https://github.com/935963004/NeuroLM.git"
UPSTREAM_COMMIT = "0cda9876d8ce6ee07ed0c43eee5e9a6f5c24b177"
UPSTREAM_ROOT = Path("/content/NeuroLM")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "neurolm/requirements-colab.txt")])
run([sys.executable, "-c", "from huggingface_hub import is_offline_mode; import transformers; print('dependency import check passed')"])
loaded_hub = sys.modules.get("huggingface_hub")
if loaded_hub is not None and not hasattr(loaded_hub, "is_offline_mode"):
    raise RuntimeError("huggingface_hub was imported before upgrade. Restart the runtime, then rerun Cell 1.")
if not (UPSTREAM_ROOT / ".git").exists():
    UPSTREAM_ROOT.mkdir(parents=True, exist_ok=True)
    run(["git", "init"], cwd=UPSTREAM_ROOT)
    run(["git", "remote", "add", "origin", UPSTREAM_URL], cwd=UPSTREAM_ROOT)
run(["git", "fetch", "--depth", "1", "origin", UPSTREAM_COMMIT], cwd=UPSTREAM_ROOT)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=UPSTREAM_ROOT)
os.chdir(PROJECT_ROOT / "neurolm")
run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=Path.cwd())
print("huggingface_hub:", package_metadata.version("huggingface_hub"))
print("transformers:", package_metadata.version("transformers"))
print("tiktoken:", package_metadata.version("tiktoken"))
print("Official NeuroLM commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip())

In [ ]:
# 2) Mount Drive and edit only these paths if your thesis layout differs.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/neurolm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/neurolm"
RAW_PACKS = CACHE_ROOT / "raw_eeg_packs_v2"
CHECKPOINT_ROOT = CACHE_ROOT / "upstream_checkpoints"
GPT2_CACHE = CACHE_ROOT / "gpt2_prompt_features_v4"
RESULTS_DIR = RESULTS_ROOT / "gpt2_verbalizer_v4"

if not (RAW_PACKS / "cache_manifest.json").exists():
    raise FileNotFoundError("Finish V2 Cell 3 first; its raw packs are V4's input")
for path in (CHECKPOINT_ROOT, GPT2_CACHE, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("V2 raw packs:", RAW_PACKS)
print("V4 GPT-2 cache:", GPT2_CACHE)
print("V4 results:", RESULTS_DIR)

In [ ]:
# 3) Rebuild the audited mapping, reuse NeuroLM-B, and initialize the complete frozen model.
import json
import torch
from huggingface_hub import hf_hub_download
from src.channels import build_mne_spatial_mapping, select_usable_mapping
from src.config import CHECKPOINT_FILENAME, CHECKPOINT_REPOSITORY
from src.gpt2_cache import OfficialNeuroLMGPT2

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")
mapping_all, mapping = select_usable_mapping(build_mne_spatial_mapping())
mapping_all.to_csv(RESULTS_DIR / "spatial_mapping.csv", index=False)
mapping_report = {
    "assignments_total": len(mapping_all),
    "assignments_used": len(mapping),
    "assignments_excluded_over_30_deg": int((~mapping_all.use_for_encoder).sum()),
    "used_mean_angular_distance_deg": float(mapping.angular_distance_deg.mean()),
    "used_max_angular_distance_deg": float(mapping.angular_distance_deg.max()),
}
(RESULTS_DIR / "spatial_mapping_diagnostics.json").write_text(json.dumps(mapping_report, indent=2))

CHECKPOINT_PATH = Path(hf_hub_download(
    repo_id=CHECKPOINT_REPOSITORY, filename=CHECKPOINT_FILENAME, local_dir=CHECKPOINT_ROOT
))
checkpoint_bytes = CHECKPOINT_PATH.stat().st_size
if checkpoint_bytes < 2_000_000_000:
    raise IOError("Checkpoint is unexpectedly small or incomplete")
encoder = OfficialNeuroLMGPT2(
    UPSTREAM_ROOT, CHECKPOINT_PATH, mapping.neurolm_index.to_numpy(),
    mapping.zuco_index.to_numpy(), device="cuda", maximum_seconds=3
)
provenance = {
    "checkpoint_repository": CHECKPOINT_REPOSITORY,
    "checkpoint_filename": CHECKPOINT_FILENAME,
    "checkpoint_bytes": checkpoint_bytes,
    "upstream_commit": UPSTREAM_COMMIT,
    "full_model_load": encoder.load_report,
    "prompt": encoder.prompt,
    "verbalizer_token_ids": encoder.verbalizer_token_ids,
}
(RESULTS_DIR / "checkpoint_provenance.json").write_text(json.dumps(provenance, indent=2))
print(json.dumps(mapping_report, indent=2))
print(f"Checkpoint: {CHECKPOINT_PATH} ({checkpoint_bytes / 1e9:.3f} GB)")
print("Prompt:", encoder.prompt)
print("Verbalizer tokens:", encoder.verbalizer_token_ids)

In [ ]:
# 4) Extract/resume one compact frozen GPT-2 prompt-state pack per subject.
from src.gpt2_cache import extract_gpt2_subject_packs

manifest = extract_gpt2_subject_packs(
    raw_pack_dir=RAW_PACKS,
    output_dir=GPT2_CACHE,
    encoder=encoder,
    overwrite=False,
    batch_size=4,
)
print(json.dumps(manifest["report"] | {"failure_rows": len(manifest["report"]["failures"])}, indent=2))
del encoder
torch.cuda.empty_cache()

In [ ]:
# 5) Load the tiny feature cache, save diagnostics, and smoke-test the locked adapter.
from src.gpt2_cache import load_gpt2_records
from src.gpt2_verbalizer import GPT2VerbalizerConfig, smoke_test_gpt2_verbalizer

records, verbalizer_vectors, recording_metadata, dataset_report = load_gpt2_records(GPT2_CACHE)
recording_metadata.to_csv(RESULTS_DIR / "recording_metadata.csv", index=False)
(RESULTS_DIR / "dataset_diagnostics.json").write_text(json.dumps(dataset_report, indent=2))
print(json.dumps(dataset_report, indent=2))

evaluation_config = GPT2VerbalizerConfig(embedding_size=dataset_report["embedding_size"])
smoke = smoke_test_gpt2_verbalizer(
    records, verbalizer_vectors, evaluation_config, device="cuda"
)
print(json.dumps(smoke, indent=2))

In [ ]:
# 6) Run/resume the locked 3-seed × 5-fold screen and save the final stoplight decision.
import matplotlib.pyplot as plt
from src.gpt2_verbalizer import evaluate_gpt2_verbalizer

metrics, predictions, summary, delta, gate = evaluate_gpt2_verbalizer(
    records=records,
    verbalizer_vectors=verbalizer_vectors,
    output_dir=RESULTS_DIR,
    dataset_fingerprint=dataset_report["dataset_fingerprint"],
    config=evaluation_config,
    device="cuda",
)
display(summary)
print(json.dumps(gate, indent=2))

plot_rows = metrics.groupby("setup").macro_f1.agg(["mean", "std"]).sort_values("mean")
ax = plot_rows["mean"].plot.barh(xerr=plot_rows["std"], figsize=(8, 4), capsize=3)
majority_macro_f1 = float(plot_rows.loc["majority", "mean"])
ax.axvline(majority_macro_f1, color="black", linestyle="--", linewidth=1, label="majority macro-F1")
ax.set_xlabel("Macro-F1 across folds")
ax.set_ylabel("")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "macro_f1_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved all V4 results:", RESULTS_DIR)

## Interpretation rule

A **green** result is eligible only for a separately locked confirmation. A **yellow** result is recorded without tuning. A **red** result ends the broad screen. The primary comparison is always the aligned adapter versus the independently trained, split-local shuffled-pairing adapter.